# Phase 1 — Augmented Backbone Comparison

**Goal:** compare ResNet50, EfficientNet-B0, and ConvNeXt-Tiny under the **same augmentation, sampling, training, validation, and evaluation protocol**.

This notebook is designed for the already cropped 224×224 face dataset:

- `train/REAL`, `train/FAKE`
- `val/REAL`, `val/FAKE`
- `test/REAL`, `test/FAKE`

### Phase 1 protocol
- Training: clean cropped faces + moderate augmentation.
- Augmentation: horizontal flip, mild color variation, JPEG compression (Q=40–100), mild Gaussian blur, mild Gaussian noise.
- Validation: **clean only**.
- Test: **HQ, JPEG Q60, JPEG Q40**.
- Models: ResNet50, EfficientNet-B0, ConvNeXt-Tiny.
- Seeds: 42, 123, 2024.
- Backbone: **fully trainable** with a smaller learning rate than the classification head.
- Selection: validation F1/AUC + HQ/LQ robustness and seed stability, not Accuracy alone.


In [ ]:
# ============================================================
# CELL 01 — ENVIRONMENT
# ============================================================

!pip install -q -U scikit-learn

import os
import random
import time
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.transforms import functional as TF
import torchvision

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

print("=" * 70)
print("PHASE 1 — AUGMENTED BACKBONE COMPARISON")
print("=" * 70)
print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CELL 02 — CONFIGURATION + REPRODUCIBILITY
# ============================================================

SEEDS = [42, 123, 2024]

DATA_DIR = Path("/content/face_frames")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 10

# Differential learning rates:
# backbone learns slowly; new classification head learns faster.
BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 4
TRAIN_SAMPLES_PER_EPOCH = 20000

# Early stopping / model selection
EARLY_STOPPING_PATIENCE = 3

# Compression augmentation
JPEG_AUG_PROB = 0.50
JPEG_Q_MIN = 40
JPEG_Q_MAX = 100

# Mild non-compression augmentations
BLUR_PROB = 0.20
NOISE_PROB = 0.20
COLOR_PROB = 0.20

# Final robustness evaluation
TEST_JPEG_QUALITIES = [60, 40]

CLASS_NAMES = ["REAL", "FAKE"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEEDS[0])

print("DATA_DIR:", DATA_DIR)
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("BACKBONE_LR:", BACKBONE_LR)
print("HEAD_LR:", HEAD_LR)
print("TRAIN_SAMPLES_PER_EPOCH:", TRAIN_SAMPLES_PER_EPOCH)
print("SEEDS:", SEEDS)
print("DEVICE:", device)

In [ ]:
# ============================================================
# CELL 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# CELL 04 — UNZIP DATASET
# ============================================================

ZIP_PATH = Path("/content/drive/MyDrive/face_frames.zip")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Dataset ZIP not found: {ZIP_PATH}\n"
        "Please update ZIP_PATH to your actual Google Drive path."
    )

# Only unzip when the target directory does not already exist.
if not DATA_DIR.exists():
    !unzip -q "$ZIP_PATH" -d /content/
    print("Dataset extracted to:", DATA_DIR)
else:
    print("Dataset directory already exists:", DATA_DIR)

In [ ]:
# ============================================================
# CELL 05 — VERIFY DATASET
# ============================================================

for name, path in {
    "TRAIN": TRAIN_DIR,
    "VAL": VAL_DIR,
    "TEST": TEST_DIR,
}.items():
    print(
        f"{name:>5}: "
        f"{'OK' if path.exists() else 'MISSING'} -> {path}"
    )

    if path.exists():
        print(
            "       Classes:",
            sorted([p.name for p in path.iterdir() if p.is_dir()])
        )

if not all(p.exists() for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]):
    raise FileNotFoundError("Train/val/test directories are not all present.")

In [ ]:
# ============================================================
# CELL 06 — DATASET STATISTICS
# Supports both:
#   train/REAL/*.jpg
# and nested structures:
#   train/REAL/video_x/*.jpg
# ============================================================

def collect_image_paths(class_dir):
    paths = []
    if not class_dir.exists():
        return paths

    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            paths.append(p)

    return sorted(paths)

def get_dataset_statistics(split_dir):
    rows = []

    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        image_paths = collect_image_paths(class_dir)

        rows.append({
            "Split": split_dir.name,
            "Class": class_name,
            "Images": len(image_paths),
        })

    return pd.DataFrame(rows)

stats = pd.concat([
    get_dataset_statistics(TRAIN_DIR),
    get_dataset_statistics(VAL_DIR),
    get_dataset_statistics(TEST_DIR),
], ignore_index=True)

display(stats)

print("\nExpected counts from your current dataset:")
print("Train: REAL 7,898 | FAKE 46,448")
print("Val:   REAL 1,020 | FAKE 5,982")
print("Test:  REAL 1,009 | FAKE 5,878")

In [ ]:
# ============================================================
# CELL 07 — DATASET CLASS
# ============================================================

class DeepfakeDataset(Dataset):

    LABEL_MAP = {
        "REAL": 0,
        "FAKE": 1,
    }

    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        for class_name, label in self.LABEL_MAP.items():
            class_dir = self.root_dir / class_name

            for image_path in collect_image_paths(class_dir):
                # Use the first parent directory above the image as a
                # lightweight identifier when nested video folders exist.
                video_id = image_path.parent.name

                self.samples.append({
                    "image_path": image_path,
                    "label": label,
                    "video_id": video_id,
                })

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found in {self.root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        with Image.open(sample["image_path"]) as img:
            image = img.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(sample["label"], dtype=torch.float32),
            "video_id": sample["video_id"],
            "image_path": str(sample["image_path"]),
        }

In [ ]:
# ============================================================
# CELL 08 — CUSTOM JPEG + NOISE TRANSFORMS
# ============================================================

class RandomJPEGCompression:
    """Randomly recompress a PIL image with JPEG quality in [min_q, max_q]."""

    def __init__(self, p=0.5, min_q=40, max_q=100):
        self.p = p
        self.min_q = min_q
        self.max_q = max_q

    def __call__(self, img):
        if random.random() > self.p:
            return img

        quality = random.randint(self.min_q, self.max_q)

        import io
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)

        with Image.open(buffer) as compressed:
            return compressed.convert("RGB")


class RandomGaussianNoise:
    """Adds mild Gaussian noise after ToTensor."""

    def __init__(self, p=0.2, std_range=(0.005, 0.025)):
        self.p = p
        self.std_range = std_range

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor

        std = random.uniform(*self.std_range)
        noise = torch.randn_like(tensor) * std
        return torch.clamp(tensor + noise, 0.0, 1.0)


class RandomMildBlur:
    """Mild Gaussian blur that does not strongly destroy facial artifacts."""

    def __init__(self, p=0.2, radius_range=(0.1, 0.8)):
        self.p = p
        self.radius_range = radius_range

    def __call__(self, img):
        if random.random() > self.p:
            return img

        radius = random.uniform(*self.radius_range)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))

In [ ]:
# ============================================================
# CELL 09 — TRAIN / VALIDATION TRANSFORMS
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# IMPORTANT:
# This is the SAME augmentation policy for all three backbones.
# Validation remains clean to make model comparison fair.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    # Geometry
    transforms.RandomHorizontalFlip(p=0.5),

    # Mild photometric variation
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.08,
            hue=0.02,
        )
    ], p=COLOR_PROB),

    # Compression robustness
    RandomJPEGCompression(
        p=JPEG_AUG_PROB,
        min_q=JPEG_Q_MIN,
        max_q=JPEG_Q_MAX,
    ),

    # Mild degradation
    RandomMildBlur(
        p=BLUR_PROB,
        radius_range=(0.1, 0.8),
    ),

    transforms.ToTensor(),

    RandomGaussianNoise(
        p=NOISE_PROB,
        std_range=(0.005, 0.025),
    ),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

print("Training augmentation:")
print("- RandomHorizontalFlip")
print("- Mild ColorJitter")
print(f"- JPEG compression: p={JPEG_AUG_PROB}, Q={JPEG_Q_MIN}–{JPEG_Q_MAX}")
print(f"- Mild GaussianBlur: p={BLUR_PROB}")
print(f"- Mild GaussianNoise: p={NOISE_PROB}")
print("\nValidation/Test: clean deterministic preprocessing only.")

In [ ]:
# ============================================================
# CELL 10 — CREATE DATASETS
# ============================================================

train_dataset = DeepfakeDataset(TRAIN_DIR, transform=train_transform)
val_dataset = DeepfakeDataset(VAL_DIR, transform=eval_transform)
test_dataset = DeepfakeDataset(TEST_DIR, transform=eval_transform)

print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))

In [ ]:
# ============================================================
# CELL 11 — CLASS DISTRIBUTION + WEIGHTED SAMPLER
# ============================================================

train_labels = np.array([
    sample["label"] for sample in train_dataset.samples
])

class_counts = np.bincount(train_labels, minlength=2)

for label, count in enumerate(class_counts):
    print(
        f"{CLASS_NAMES[label]:>5}: "
        f"{count:,} ({count / len(train_labels):.2%})"
    )

# Balanced sampling prevents the ~1:6 REAL/FAKE imbalance from
# dominating the optimization objective.
class_weights = 1.0 / class_counts

sample_weights = torch.tensor(
    [class_weights[label] for label in train_labels],
    dtype=torch.double,
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=TRAIN_SAMPLES_PER_EPOCH,
    replacement=True,
)

print("\nWeightedRandomSampler: READY")
print("Samples per epoch:", TRAIN_SAMPLES_PER_EPOCH)

In [ ]:
# ============================================================
# CELL 12 — DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))

In [ ]:
# ============================================================
# CELL 13 — SANITY CHECK AUGMENTATION
# ============================================================

# Visual inspection is important before spending GPU time.
# Denormalize images for display.

def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1)

batch = next(iter(train_loader))
images = batch["image"][:8]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax, img in zip(axes.ravel(), images):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.axis("off")

plt.suptitle("Examples from Phase 1 Training Augmentation")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 14 — MODEL FACTORY
# ============================================================

def create_model(model_name):
    if model_name == "resnet50":
        model = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, 1)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, 1)

    elif model_name == "convnext_tiny":
        model = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT
        )
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, 1)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


MODEL_NAMES = [
    "resnet50",
    "efficientnet_b0",
    "convnext_tiny",
]

print("Models:")
for name in MODEL_NAMES:
    print("-", name)

In [ ]:
# ============================================================
# CELL 15 — DIFFERENTIAL LEARNING-RATE PARAMETER GROUPS
# ============================================================

def get_parameter_groups(model_name, model):
    """
    Fully train the pretrained backbone, but use:
      backbone LR = 1e-4
      classification head LR = 1e-3

    This is standard fine-tuning for Phase 1.
    It is NOT the deep hyperparameter optimization reserved for Phase 4.
    """

    if model_name == "resnet50":
        head_params = list(model.fc.parameters())

    elif model_name == "efficientnet_b0":
        head_params = list(model.classifier.parameters())

    elif model_name == "convnext_tiny":
        head_params = list(model.classifier.parameters())

    else:
        raise ValueError(model_name)

    head_ids = {id(p) for p in head_params}

    backbone_params = [
        p for p in model.parameters()
        if p.requires_grad and id(p) not in head_ids
    ]

    return [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
        },
    ]


def inspect_model(model_name):
    model = create_model(model_name)

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    print(f"\n{model_name}")
    print(f"Total params     : {total:,}")
    print(f"Trainable params : {trainable:,}")
    print(f"Trainable ratio  : {100 * trainable / total:.2f}%")


for name in MODEL_NAMES:
    inspect_model(name)

In [ ]:
# ============================================================
# CELL 16 — TRAIN / EVALUATION UTILITIES
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()

    running_loss = 0.0
    num_samples = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = images.size(0)
        running_loss += loss.item() * bs
        num_samples += bs

    return running_loss / num_samples


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    num_samples = 0

    all_labels = []
    all_probs = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast(
            "cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        bs = images.size(0)
        running_loss += loss.item() * bs
        num_samples += bs

        probs = torch.sigmoid(logits)

        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    labels = np.asarray(all_labels)
    probs = np.asarray(all_probs)
    predictions = (probs >= 0.5).astype(int)

    return {
        "loss": running_loss / num_samples,
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "recall": recall_score(
            labels, predictions, zero_division=0
        ),
        "f1": f1_score(
            labels, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(labels, probs),
    }

In [ ]:
# ============================================================
# CELL 17 — TRAIN ONE MODEL
# ============================================================

def train_model(
    model_name,
    seed,
    num_epochs=NUM_EPOCHS,
):
    set_seed(seed)

    print("\n" + "=" * 80)
    print(f"TRAINING: {model_name} | SEED: {seed}")
    print("=" * 80)

    model = create_model(model_name).to(device)

    # All backbone parameters remain trainable.
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        get_parameter_groups(model_name, model),
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=1,
    )

    best_f1 = -np.inf
    best_auc = -np.inf
    best_epoch = 0
    best_state = None
    history = []
    patience_counter = 0

    total_start = time.time()

    for epoch in range(num_epochs):
        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
        )

        scheduler.step(val_metrics["f1"])

        epoch_time = time.time() - epoch_start

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "backbone_lr": optimizer.param_groups[0]["lr"],
            "head_lr": optimizer.param_groups[1]["lr"],
            "epoch_time_sec": epoch_time,
        })

        print(
            f"Epoch [{epoch+1:02d}/{num_epochs}] | "
            f"Loss {train_loss:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val AUC {val_metrics['roc_auc']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Time {epoch_time:.1f}s"
        )

        # Primary checkpoint criterion = validation F1.
        # AUC is retained as a secondary criterion.
        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_auc = val_metrics["roc_auc"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

    total_time = time.time() - total_start

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)

    print(
        f"\nBest epoch: {best_epoch} | "
        f"Best Val F1: {best_f1:.4f} | "
        f"Val AUC: {best_auc:.4f}"
    )
    print(f"Total time: {total_time / 60:.2f} min")

    return model, history_df

In [ ]:
# ============================================================
# CELL 18 — PILOT RUN
# ============================================================

# Run this first to verify the entire pipeline before launching
# 3 models × 3 seeds.
#
# pilot_model, pilot_history = train_model(
#     model_name="resnet50",
#     seed=42,
#     num_epochs=1,
# )
#
# display(pilot_history)

In [ ]:
# ============================================================
# CELL 19 — RUN 3 MODELS × 3 SEEDS
# ============================================================

MODEL_NAMES = [
    "resnet50",
    "efficientnet_b0",
    "convnext_tiny",
]

all_results = []
trained_models = {}
histories = {}

for seed in SEEDS:
    for model_name in MODEL_NAMES:

        model, history = train_model(
            model_name=model_name,
            seed=seed,
            num_epochs=NUM_EPOCHS,
        )

        best_idx = history["val_f1"].idxmax()
        best_row = history.loc[best_idx]

        key = f"{model_name}__seed{seed}"
        trained_models[key] = model
        histories[key] = history

        all_results.append({
            "model": model_name,
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),
            "val_accuracy": float(best_row["val_accuracy"]),
            "val_precision": float(best_row["val_precision"]),
            "val_recall": float(best_row["val_recall"]),
            "val_f1": float(best_row["val_f1"]),
            "val_roc_auc": float(best_row["val_roc_auc"]),
        })

        print("\nCompleted:", key)

results_df = pd.DataFrame(all_results)

print("\n" + "=" * 80)
print("ALL 9 EXPERIMENTS COMPLETED")
print("=" * 80)

display(results_df)

In [ ]:
# ============================================================
# CELL 20 — PHASE 1 VALIDATION SUMMARY: MEAN ± STD
# ============================================================

def mean_std(series):
    return f"{series.mean():.4f} ± {series.std(ddof=1):.4f}"

validation_summary = results_df.groupby("model").agg(
    Accuracy=("val_accuracy", mean_std),
    Precision=("val_precision", mean_std),
    Recall=("val_recall", mean_std),
    F1=("val_f1", mean_std),
    ROC_AUC=("val_roc_auc", mean_std),
)

display(validation_summary)

In [ ]:
# ============================================================
# CELL 21 — CREATE JPEG TEST SET ON-THE-FLY
# ============================================================

class JPEGTestDataset(Dataset):
    """Deterministic JPEG-compressed version of the same test images."""

    def __init__(self, base_dataset, quality):
        self.base_dataset = base_dataset
        self.quality = quality

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        sample = self.base_dataset.samples[index]

        with Image.open(sample["image_path"]) as img:
            image = img.convert("RGB")

        import io
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0)

        with Image.open(buffer) as compressed:
            image = compressed.convert("RGB")

        image = eval_transform(image)

        return {
            "image": image,
            "label": torch.tensor(
                sample["label"],
                dtype=torch.float32
            ),
            "video_id": sample["video_id"],
            "image_path": str(sample["image_path"]),
        }


def make_test_loader(quality=None):
    if quality is None:
        dataset = test_dataset
    else:
        dataset = JPEGTestDataset(
            test_dataset,
            quality=quality
        )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )


test_loaders = {
    "HQ": make_test_loader(None),
    "JPEG_Q60": make_test_loader(60),
    "JPEG_Q40": make_test_loader(40),
}

print({
    name: len(loader.dataset)
    for name, loader in test_loaders.items()
})

In [ ]:
# ============================================================
# CELL 22 — TEST ALL 9 TRAINED MODELS ON HQ / Q60 / Q40
# ============================================================

test_results = []

criterion = nn.BCEWithLogitsLoss()

for key, model in trained_models.items():

    model_name, seed_text = key.split("__seed")
    seed = int(seed_text)

    for condition, loader in test_loaders.items():

        metrics = evaluate(
            model,
            loader,
            criterion,
        )

        test_results.append({
            "model": model_name,
            "seed": seed,
            "test_condition": condition,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"],
        })

test_results_df = pd.DataFrame(test_results)

display(
    test_results_df.sort_values(
        ["model", "seed", "test_condition"]
    )
)

In [ ]:
# ============================================================
# CELL 23 — COMPRESSION ROBUSTNESS SUMMARY
# ============================================================

robustness = test_results_df.pivot_table(
    index=["model", "seed"],
    columns="test_condition",
    values="accuracy",
).reset_index()

for condition in ["JPEG_Q60", "JPEG_Q40"]:
    robustness[f"Drop_HQ_to_{condition}"] = (
        robustness["HQ"] - robustness[condition]
    )

display(robustness)

robustness_summary = robustness.groupby("model").agg(
    HQ_Accuracy=("HQ", mean_std),
    Q60_Accuracy=("JPEG_Q60", mean_std),
    Q40_Accuracy=("JPEG_Q40", mean_std),
    Drop_HQ_to_Q60=("Drop_HQ_to_JPEG_Q60", mean_std),
    Drop_HQ_to_Q40=("Drop_HQ_to_JPEG_Q40", mean_std),
)

display(robustness_summary)

In [ ]:
# ============================================================
# CELL 24 — FINAL PHASE 1 COMPARISON TABLE
# ============================================================

# Combine clean validation performance with compression robustness.
final_phase1 = validation_summary.copy()

final_phase1 = final_phase1.join(
    robustness_summary
)

display(final_phase1)

In [ ]:
# ============================================================
# CELL 25 — SELECT CANDIDATE BACKBONE
# ============================================================

# IMPORTANT:
# Do not select using Accuracy alone.
# We rank by:
#   1) Validation F1
#   2) Validation ROC-AUC
#   3) Q40 accuracy
#   4) Smaller HQ→Q40 accuracy drop
#
# This is a screening rule for Phase 1, not final scientific tuning.

selection = (
    results_df.groupby("model")
    .agg(
        mean_val_f1=("val_f1", "mean"),
        mean_val_auc=("val_roc_auc", "mean"),
    )
    .join(
        robustness.groupby("model")
        .agg(
            mean_hq=("HQ", "mean"),
            mean_q40=("JPEG_Q40", "mean"),
            mean_drop_q40=("Drop_HQ_to_JPEG_Q40", "mean"),
        )
    )
)

selection = selection.sort_values(
    by=[
        "mean_val_f1",
        "mean_val_auc",
        "mean_q40",
        "mean_drop_q40",
    ],
    ascending=[False, False, False, True],
)

display(selection)

best_backbone = selection.index[0]

print("=" * 70)
print("PHASE 1 CANDIDATE BACKBONE")
print("=" * 70)
print(best_backbone)
print("\nThis is the candidate for Phase 4 fine-tuning.")

In [ ]:
# ============================================================
# CELL 26 — SAVE RESULTS
# ============================================================

OUTPUT_DIR = Path("/content/phase1_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_DIR / "phase1_validation_raw_results.csv",
    index=False
)

validation_summary.to_csv(
    OUTPUT_DIR / "phase1_validation_mean_std.csv"
)

test_results_df.to_csv(
    OUTPUT_DIR / "phase1_test_hq_q60_q40_raw.csv",
    index=False
)

robustness.to_csv(
    OUTPUT_DIR / "phase1_robustness_by_seed.csv",
    index=False
)

robustness_summary.to_csv(
    OUTPUT_DIR / "phase1_robustness_mean_std.csv"
)

final_phase1.to_csv(
    OUTPUT_DIR / "phase1_final_comparison.csv"
)

selection.to_csv(
    OUTPUT_DIR / "phase1_backbone_selection.csv"
)

print("Saved Phase 1 results to:", OUTPUT_DIR)
print("\nFiles:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

In [ ]:
# ============================================================
# CELL 27 — OPTIONAL: SAVE BEST CHECKPOINTS
# ============================================================

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Save the best seed for each backbone according to validation F1.
best_seed_rows = (
    results_df.sort_values("val_f1", ascending=False)
    .groupby("model")
    .head(1)
)

for _, row in best_seed_rows.iterrows():
    key = f"{row['model']}__seed{int(row['seed'])}"
    model = trained_models[key]

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{row['model']}_seed{int(row['seed'])}_phase1.pt"
    )

    torch.save(
        {
            "model_name": row["model"],
            "seed": int(row["seed"]),
            "state_dict": model.state_dict(),
            "val_f1": float(row["val_f1"]),
            "val_roc_auc": float(row["val_roc_auc"]),
        },
        checkpoint_path,
    )

    print("Saved:", checkpoint_path)

## Phase 1 interpretation

The important outputs are:

1. **`phase1_validation_mean_std.csv`** — clean validation comparison.
2. **`phase1_robustness_mean_std.csv`** — HQ vs JPEG Q60/Q40 robustness.
3. **`phase1_final_comparison.csv`** — combined table.
4. **`phase1_backbone_selection.csv`** — candidate backbone for the next stage.

### What Phase 1 does NOT do

- It does not perform deep hyperparameter tuning.
- It does not perform Grad-CAM.
- It does not claim the selected backbone is the final model.
- It does not use JPEG-compressed test images for training or validation.

The selected backbone becomes the candidate for the dedicated fine-tuning phase.
